In [1]:
import pandas as pd
import numpy as np

In [2]:
pop_size = pd.DataFrame({
    'InfState': ['S', 'S', 'I', 'I', 'S', 'S', 'I', 'I', 'S', 'S', 'I', 'I'],
    'County': pd.Categorical(['Orange_HA', 'Orange_LA', 'Orange_HA', 'Orange_LA', 
                              'Durham_HA', 'Durham_LA', 'Durham_HA', 'Durham_LA', 
                              'Chatham_HA', 'Chatham_LA', 'Chatham_HA', 'Chatham_LA'], 
                             categories=['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']),
    'N': [5000, 5000, 8000, 316, 40000, 50000, 2000, 280, 10000, 20000, 8000, 266],
    'T': 2025
})
pop_size

,InfState,County,N,T
0,S,Orange_HA,5000,2025
1,S,Orange_LA,5000,2025
2,I,Orange_HA,8000,2025
3,I,Orange_LA,316,2025
4,S,Durham_HA,40000,2025
5,S,Durham_LA,50000,2025
6,I,Durham_HA,2000,2025
7,I,Durham_LA,280,2025
8,S,Chatham_HA,10000,2025
9,S,Chatham_LA,20000,2025


In [3]:
#contact rate matrix, Beta: High Activity, Low Activity
contact_rate_matrix = np.array([[0.1428, 0.0049], [0.0055, 0.0077]])
contact_rate_matrix

array([[0.1428, 0.0049],
       [0.0055, 0.0077]])

In [4]:
#Geographcial mixing matrix, T: Orange, Durham, Chatham
geo_mixing_matrix = np.array([[0.7, 0.1869, 0.1131], [0.1851, 0.7, 0.1150], [0.2210, 0.07895, 0.7]])
geo_mixing_matrix

array([[0.7    , 0.1869 , 0.1131 ],
       [0.1851 , 0.7    , 0.115  ],
       [0.221  , 0.07895, 0.7    ]])

In [5]:
transmission_by_geo = np.kron(geo_mixing_matrix, contact_rate_matrix)
transmission_by_geo = np.round(transmission_by_geo, decimals=5)
transmission_by_geo

array([[0.09996, 0.00343, 0.02669, 0.00092, 0.01615, 0.00055],
       [0.00385, 0.00539, 0.00103, 0.00144, 0.00062, 0.00087],
       [0.02643, 0.00091, 0.09996, 0.00343, 0.01642, 0.00056],
       [0.00102, 0.00143, 0.00385, 0.00539, 0.00063, 0.00089],
       [0.03156, 0.00108, 0.01127, 0.00039, 0.09996, 0.00343],
       [0.00122, 0.0017 , 0.00043, 0.00061, 0.00385, 0.00539]])

In [6]:
index_names= ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
column_names = ['Orange_HA', 'Orange_LA', 'Durham_HA', 'Durham_LA', 'Chatham_HA', 'Chatham_LA']
pd.DataFrame(transmission_by_geo, index=index_names, columns=column_names)

,Orange_HA,Orange_LA,Durham_HA,Durham_LA,Chatham_HA,Chatham_LA
Orange_HA,0.09996,0.00343,0.02669,0.00092,0.01615,0.00055
Orange_LA,0.00385,0.00539,0.00103,0.00144,0.00062,0.00087
Durham_HA,0.02643,0.00091,0.09996,0.00343,0.01642,0.00056
Durham_LA,0.00102,0.00143,0.00385,0.00539,0.00063,0.00089
Chatham_HA,0.03156,0.00108,0.01127,0.00039,0.09996,0.00343
Chatham_LA,0.00122,0.00170,0.00043,0.00061,0.00385,0.00539


In [7]:
#need to group by both inf_col and gorup_col to separate HA and LA data
inf_array = pop_size.loc[pop_size['InfState']=='I'].groupby(['County'], observed=False)['N'].sum(numeric_only=True).values
inf_array

array([8000,  316, 2000,  280, 8000,  266])

In [8]:
prI = np.power(np.exp(-1 * transmission_by_geo), inf_array)
prI

array([[0.00000000e+000, 3.38280448e-001, 6.56690231e-024,
        7.72904332e-001, 7.74734575e-057, 8.63898495e-001],
       [4.20465104e-014, 1.82092587e-001, 1.27453970e-001,
        6.68178450e-001, 7.01292783e-003, 7.93406165e-001],
       [1.48858880e-092, 7.50091560e-001, 1.49915721e-087,
        3.82739759e-001, 8.93463586e-058, 8.61603578e-001],
       [2.85862395e-004, 6.36430537e-001, 4.52827183e-004,
        2.21086777e-001, 6.47374832e-003, 7.89196452e-001],
       [2.23526598e-110, 7.10859840e-001, 1.62555766e-010,
        8.96551089e-001, 0.00000000e+000, 4.01567356e-001],
       [5.77146221e-005, 5.84382234e-001, 4.23162082e-001,
        8.42990155e-001, 4.20465104e-014, 2.38415578e-001]])

In [9]:
prI = 1 - prI.prod(axis=1)
prI

array([1., 1., 1., 1., 1., 1.])

In [10]:
is_susceptible = (pop_size['InfState'] == 'S')
is_susceptible

0      True
1      True
2     False
3     False
4      True
5      True
6     False
7     False
8      True
9      True
10    False
11    False
Name: InfState, dtype: bool

In [11]:
deltas = pop_size[is_susceptible].copy()
deltas

,InfState,County,N,T
0,S,Orange_HA,5000,2025
1,S,Orange_LA,5000,2025
4,S,Durham_HA,40000,2025
5,S,Durham_LA,50000,2025
8,S,Chatham_HA,10000,2025
9,S,Chatham_LA,20000,2025


In [12]:
present_category_codes = pop_size['County'].cat.codes.to_numpy()
present_category_codes

array([0, 1, 0, 1, 2, 3, 2, 3, 4, 5, 4, 5], dtype=int8)

In [13]:
susceptible_group_codes = present_category_codes[is_susceptible.to_numpy()]
susceptible_group_codes

array([0, 1, 2, 3, 4, 5], dtype=int8)

In [14]:
prI_per_group = prI[susceptible_group_codes]
prI_per_group

array([1., 1., 1., 1., 1., 1.])

In [15]:
deltas['N'] = -deltas['N'] * prI_per_group
deltas

,InfState,County,N,T
0,S,Orange_HA,-5000.000000,2025
1,S,Orange_LA,-5000.000000,2025
4,S,Durham_HA,-40000.000000,2025
5,S,Durham_LA,-49999.999995,2025
8,S,Chatham_HA,-10000.000000,2025
9,S,Chatham_LA,-20000.000000,2025
